# Course notebook — Knowledge and Retrieval Systems

Build grounded knowledge systems whose ingestion, retrieval, generation, and citations can be evaluated separately.

Use Python 3.10 or later. All inputs are included and the workbook runs without API keys. Each lecture demonstration is a small model of the lecture mechanism. Run it, compare the result with your prediction, and use the section checkpoint to record what you learned.


## Practice case · The policy that changed on Monday

Imani answers employee reimbursement questions. On Friday the limit is $80. A policy update makes it $60 on Monday and adds an exception for approved field work. Old copies are still in email attachments. Imani wants one answer that shows its evidence and explains which rule applied when a purchase was made.

The workplace scenario is fictional.


## Knowledge strategy and ingestion


### Lecture 01 · Knowledge Outside the Model

Decide among direct context, search, databases, knowledge graphs, RAG, fine-tuning, and deterministic rules.

#### A policy is state that can change

An employee made a purchase on Sunday and asks on Tuesday. The latest policy alone gives the wrong number for one of those dates.

Persist effective dates outside the model and resolve which revision applies to the requested event. Hold the question fixed and change the event date. The evidence selection should change even if the model and prompt stay identical.

#### Read the mechanism

**knowledge freshness**

Knowledge freshness measures how current stored facts are relative to the real world. Stale indexes cause confident wrong answers until re-ingestion catches up.

A travel policy updated yesterday is invisible if the index last synced last month.

**grounding**

Grounding ties model statements to verifiable evidence—retrieved passages, database rows, tool outputs. Ungrounded generation is speculation presented as fact.

Support answers should quote the ticket macro article that authorizes the refund step.

**structured data**

Structured data lives in tables, APIs, and graphs with typed fields—better for precise queries than prose retrieval. Hybrid systems route quantitative questions to SQL, not RAG alone.

'How many open P1 incidents?' needs a database query, not semantic search over runbooks.

**retrieval**

Retrieval selects candidate evidence from a corpus given a query before ranking and generation. It is candidate generation under relevance and policy constraints—not the final answer.

Hybrid retrieval returns 20 chunks for reranking; generation never sees the full million-document index.

**fine-tuning**

Fine-tuning adapts pretrained weights with supervised or preference data when prompts and RAG cannot stabilize behavior. It trades generality and ops simplicity for targeted changes.

Support tone and escalation policy may need SFT when prompts drift across thousands of ticket types.

#### Worked example

**Situation:** An enterprise assistant must answer from authorized policies and cite exact passages—product debates RAG vs fine-tuning vs bigger context.

**Baseline:** Stuff entire policy PDF into prompt—expensive and still stale.

**Application:** Classify ten requirements (freshness, authorization, structured lookup, style, math) to correct mechanism: retrieval, SQL, tools, fine-tune, or rules; document governance for each.

**Test cases:** (1) Normal: weekly-updated FAQ. (2) Boundary: numeric entitlement needing database query. (3) Adversarial: requirement for legally provable citation from signed PDF page.

**Measurement:** Correct mechanism assignment vs expert review; projected cost and freshness SLA per choice.

**Design question:** Which requirement forces retrieval even if fine-tuning improves tone?

**Predict before running.** Put knowledge in the component best suited to update, govern, query, and verify it. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.1: Knowledge Outside the Model"""

CHAPTER = "6.1"
print("chapter hook:", CHAPTER)
requirements = [
    ("cite exact page", "RAG"),
    ("friendly tone", "prompt/finetune"),
    ("live headcount", "SQL tool"),
]
for req, mechanism in requirements:
    print({"requirement": req, "mechanism": mechanism})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Classify ten requirements by the correct knowledge mechanism.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why put knowledge in the component best suited to update and verify it?
2. When does fine-tuning fail freshness requirements?
3. What baseline uses only larger context windows?

<details><summary>Check your reasoning</summary>

Q1: Misplaced knowledge breaks governance and update paths. Q2: Weights lag policy changes; cannot cite reliably. Q3: Whole-corpus prompt stuffing without retrieval.

</details>


### Lecture 02 · Document Ingestion

Preserve provenance while parsing documents, OCR, tables, images, metadata, permissions, versions, and deletions.

#### A chunk starts halfway through an exception

The parser separates ‘except for approved field work’ from the limit it qualifies. Staff get a clean passage that omits the condition.

Retain document hierarchy and provenance while chunking. Test retrieval of a paragraph containing both the limit and its exception, then confirm that the assembly step keeps them together.

#### Read the mechanism

**parsing**

Parsing converts documents—PDF, HTML, DOCX—into clean text and structure for indexing. Bad parsing loses tables, headings, and lists that retrieval cannot recover.

OCR garbling a table of limits makes correct retrieval impossible regardless of embedding quality.

**OCR**

OCR extracts text from scanned images and photos, introducing recognition errors that propagate to chunks and answers. Confidence scores help gate low-quality extractions.

Scanned contracts with skewed pages need deskew preprocessing before OCR.

**chunking**

Chunking splits documents into index units sized for retrieval precision and generation context. Boundaries should respect sections, not arbitrary token counts alone.

Splitting mid-table separates headers from values, producing useless retrieval hits.

**metadata**

Metadata tags documents with tenant, date, author, permissions, and type for filtering and ranking. Rich metadata enables policy enforcement beyond vector similarity.

Filtering by effective_date prevents superseded policies from ranking above current ones.

**provenance**

Provenance for generated media records model, prompt, timestamp, and user for copyright and authenticity disputes.

C2PA metadata embeds creation tool and prompt hash in exported campaign image.

#### Worked example

**Situation:** Policy PDFs include tables, footnotes, and permission labels; bad ingestion loses rows counsel needs for audits.

**Baseline:** Plain-text dump of PDF—tables collapse, page numbers lost.

**Application:** Build ingestion manifest tracking source URI, checksum, parse method, OCR confidence, chunk boundaries, ACL labels; measure field recovery on table-heavy pages.

**Test cases:** (1) Normal: digital PDF with text layer. (2) Boundary: scanned page OCR 85% confidence. (3) Adversarial: document marked confidential ingested into general index.

**Measurement:** Table cell recovery F1, provenance completeness score, ACL leak count (must be zero).

**Design question:** Which metadata field prevents a confidential chunk appearing in general retrieval?

**Predict before running.** Retrieval cannot recover content or permissions lost during ingestion. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.2: Document Ingestion"""

manifest = {
    "doc_id": "POL-441",
    "pages": 12,
    "acl": "HR-ONLY",
    "chunks": [{"id": 1, "page": 3, "text_hash": "abc123"}],
}
def allowed(chunk, user_groups):
    required = manifest["acl"]
    return required in user_groups
user = {"groups": ["ALL-STAFF"]}
print({"access": allowed(manifest["chunks"][0], user["groups"])})


#### Interpret and apply

Create an ingestion manifest and measure parse fidelity.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why cannot retrieval recover permissions lost during ingestion?
2. How would collapsed tables cause citation failures?
3. What ingestion baseline ignores provenance?

<details><summary>Check your reasoning</summary>

Q1: ACLs must attach to chunks at index time. Q2: Answers cite wrong numeric entitlements from garbled rows. Q3: strip-all-metadata text dump.

</details>


**Section checkpoint.** Sketch source identifiers and chunk lineage for two versions of the same policy.


## Retrieval and reranking


### Lecture 03 · Retrieval

Compare lexical, dense, sparse, hybrid, filtered, multi-query, parent-child, and late-interaction retrieval.

#### The right amount for the wrong expense

Searching for ‘field allowance’ returns a travel note because it contains both words. The reimbursement policy uses ‘approved field work’.

Compare lexical, dense, and filtered retrieval for this labeled query. Look at top-K recall before changing generation. If the necessary policy never arrives, a more fluent model can only conceal the upstream failure.

#### Read the mechanism

**BM25**

BM25 ranks documents by weighted term frequency with length normalization and term-frequency saturation. Extra keyword repetition helps less over time compared to raw TF.

A policy ID in the query should rank the exact section above generic overview pages.

**dense retrieval**

Dense retrieval embeds queries and documents into the same vector space and returns nearest neighbors by similarity.

A query about 'application unavailable' retrieves 'service is down' without lexical overlap.

**hybrid search**

Hybrid search combines lexical and dense signals—often via reciprocal rank fusion—when neither alone covers identifiers and paraphrases.

Fusion surfaces policy IDs lexically while keeping semantic matches for informal phrasing.

**query rewriting**

Query rewriting transforms requests via expansion, decomposition, or HyDE before retrieval to close vocabulary gaps.

Expanding 'PTO' to 'paid time off' helps lexical retrievers match handbook language.

**parent-child retrieval**

Parent–child retrieval indexes small child chunks for precision but returns parent sections for generation context.

A child bullet may lack the section title needed for a correct answer unless parent is joined.

#### Worked example

**Situation:** The enterprise assistant must retrieve authorized policy passages for hybrid employee queries mixing IDs and natural language.

**Baseline:** BM25 only—misses paraphrases; dense only—misses policy numbers.

**Application:** Implement lexical and vector baselines, compute recall@k on labeled set, add query rewriting and parent-child chunk retrieval for long policies.

**Test cases:** (1) Normal: "PTO accrual cap 240." (2) Boundary: parent doc updated but child chunks stale. (3) Adversarial: retrieved doc user lacks ACL to read.

**Measurement:** Recall@5 and MRR per query type; ACL-filtered recall (should exclude forbidden docs entirely).

**Design question:** When does parent-child retrieval beat flat chunking on update frequency?

**Predict before running.** Retrieval is candidate selection under relevance and policy constraints. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.3: Retrieval"""

CHAPTER = "6.3"
print("chapter hook:", CHAPTER)
docs = {"a": "PTO accrual cap is 240 hours", "b": "Leave policy overview"}
query = set("pto cap".split())
scores = {k: len(query & set(v.lower().split())) for k, v in docs.items()}
print("bm25_proxy:", sorted(scores.items(), key=lambda x: -x[1]))
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Implement lexical and vector baselines and calculate recall@k.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why is retrieval candidate selection under policy constraints?
2. How does hybrid search help policy number plus paraphrase queries?
3. What single-channel baseline should hybrid beat?

<details><summary>Check your reasoning</summary>

Q1: Must enforce relevance and authorization before generation. Q2: Lexical hits IDs, dense hits paraphrase—fusion covers both. Q3: BM25-only or dense-only on labeled 30-query set.

</details>


### Lecture 04 · Ranking and Context Selection

Use fusion, reranking, diversity, deduplication, compression, and token-aware packing.

#### Reranking must keep the date attached

The current policy ranks first for relevance, but the employee’s purchase predates it. The reranker rewards text similarity and ignores effective dates.

Enforce eligibility before comparing relevance. Keep temporal and access checks as auditable filters, then compare rank quality only among eligible passages. This makes it easier to distinguish a missing candidate from a scoring error.

#### Read the mechanism

**reciprocal rank fusion**

Reciprocal rank fusion merges ranked lists by summing 1/(k + rank) per document across retrievers.

A document ranked third lexically and second densely outscores a single-list winner.

**rerankers**

Rerankers rescore top-k candidates with cross-attention models more accurate than bi-encoders alone. They add latency proportional to candidates rescored.

Cross-encoder reranking top-50 BM25 hits improves precision@5 for policy QA.

**diversity**

Diversity in context selection avoids redundant passages that waste tokens on repeated facts. Maximal marginal relevance is a common heuristic.

Three chunks saying the same PTO limit add no value; one plus related exceptions is better.

**deduplication**

Deduplication removes near-duplicate training examples that inflate metrics and memorization.

Duplicate FAQ pairs in SFT data cause verbatim regurgitation in deployment.

**context packing**

Context packing fits selected passages into the token window respecting priority, citation needs, and truncation rules. Packing order affects what the model emphasizes.

Place highest-scored evidence first when middle-context attention is weaker in long windows.

#### Worked example

**Situation:** Initial retrieval returns twenty chunks but the model only accepts four; ranking and packing determine answer quality.

**Baseline:** Take top-4 by BM25 score—redundant sections crowd out diversity.

**Application:** Apply reciprocal rank fusion across retrievers, cross-encoder rerank, MMR diversity, deduplicate near-identical chunks, token-aware packer respecting citation metadata.

**Test cases:** (1) Normal: diverse relevant sections. (2) Boundary: token budget fits exactly three chunks. (3) Adversarial: near-duplicate chunks from template boilerplate flooding top ranks.

**Measurement:** Answer faithfulness vs number of chunks packed; latency added by reranker; redundancy rate in context.

**Design question:** When does adding more retrieved context reduce answer quality?

**Predict before running.** Every selected passage competes for limited attention; more context can reduce quality. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.4: Ranking and Context Selection"""

rank_a = ["doc-leave", "doc-expense", "doc-security"]
rank_b = ["doc-expense", "doc-leave", "doc-onboarding"]
def rrf(lists, k=60):
    scores = {}
    for ranking in lists:
        for rank, doc in enumerate(ranking, 1):
            scores[doc] = scores.get(doc, 0) + 1/(k+rank)
    return sorted(scores.items(), key=lambda x: -x[1])
print("rrf top2:", rrf([rank_a, rank_b])[:2])


#### Interpret and apply

Add reranking and measure quality versus latency.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why does every selected passage compete for limited attention?
2. How does reranking differ from fusion?
3. What baseline packs top-k without deduplication?

<details><summary>Check your reasoning</summary>

Q1: Context window and model confusion limit useful evidence. Q2: Fusion merges lists; rerank scores query-passage pairs deeply. Q3: First-k BM25 hits verbatim.

</details>


**Section checkpoint.** Compare two ranked lists and trace how reciprocal rank fusion combines them.


## Grounded answers and enterprise constraints


### Lecture 05 · RAG Generation and Citations

Construct grounded prompts, handle missing evidence, attribute claims, validate citations, and avoid unsupported synthesis.

#### A citation can point to the wrong paragraph

The answer quotes $80 and cites the correct document title. The linked paragraph is from the new revision and says $60.

Verify the claim against the exact passage and revision. If the retrieved evidence does not establish what applied on Sunday, make that uncertainty visible and hand the case to Imani.

##### Debug the answer in three passes

The purchase happened on Sunday. Imani opens the trace and finds that retrieval returned only the Monday revision. Adding stricter wording to the prompt will not restore evidence the model never received. She writes three checks.

1. **Temporal retrieval** · The same query returns the Friday revision for Sunday and the Monday revision for Tuesday. Record the sources before generation.
2. **Claim support** · Both amounts appear in the corpus. Before showing an answer, verify that the exact quoted passage supports the amount and the effective date together.
3. **Handoff** · In a broken index with only the Monday document, the assistant says it cannot confirm the Sunday limit and gives the employee a route to staff review.

These checks isolate a missing source, a wrong conclusion, and a poor fallback. The learner can now test each change separately. Change the case so that the purchase includes approved field work. Which additional evidence must the answer include before it can cite the exception?

#### Read the mechanism

**grounded generation**

Grounded generation conditions answers strictly on provided evidence, refusing when support is insufficient. Prompts and validators enforce cite-or-abstain behavior.

The model quotes section 4.2 for refund rules instead of inventing a 30-day window.

**abstention**

Abstention lets a system refuse or defer when confidence is insufficient, routing cases to humans or safer paths. It prevents forced wrong answers on ambiguous inputs.

A benefits bot abstains on incomplete forms instead of guessing eligibility that triggers appeals.

**citation precision**

Citation precision measures whether cited sources actually support the adjacent claims. Wrong citations destroy trust faster than no citations.

Linking a harassment policy to answer a parking question is high-recall citation but zero precision.

**faithfulness**

Faithfulness checks that generated statements are entailed by retrieved evidence, not hallucinated additions. It is separate from fluency or user satisfaction.

Correct tone but wrong deductible amount is unfaithful despite readable prose.

**answer validation**

Answer validation runs programmatic checks—schema, arithmetic, citation alignment—on model outputs before display. It catches errors sampling alone misses.

Verify cited policy IDs exist and quoted numbers match source tables.

#### Worked example

**Situation:** Legal requires every assistant answer to cite the exact policy passage; unsupported synthesis triggers compliance review.

**Baseline:** Model adds footnotes that do not match retrieved text.

**Application:** Ground generation on packed context only, abstain when evidence insufficient, validate each claim against cited chunk with string overlap and entailment check, reject misaligned citations.

**Test cases:** (1) Normal: answer fully supported by one chunk. (2) Boundary: partial support—should qualify or abstain. (3) Adversarial: model cites correct doc but claims opposite meaning.

**Measurement:** Citation precision/recall, faithfulness score, abstention rate on unanswerable queries.

**Design question:** What validator catches correct-doc wrong-claim citations?

**Predict before running.** A citation is useful only when it supports the nearby claim and resolves to source evidence. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.5: RAG Generation and Citations"""

CHAPTER = "6.5"
print("chapter hook:", CHAPTER)
claim = "PTO cap is 300 hours"
source = "PTO accrual cap is 240 hours"
claim_tokens = set(claim.lower().split())
source_tokens = set(source.lower().split())
overlap = len(claim_tokens & source_tokens) / len(claim_tokens)
print({"overlap": round(overlap, 2), "supported": "240" in source and "240" in claim})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Build a citation validator that checks claim-to-source alignment.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. When is a citation useful versus decorative?
2. How should the system behave with missing evidence?
3. What generation baseline skips citation validation?

<details><summary>Check your reasoning</summary>

Q1: When nearby claim is entailed by cited span with correct pointer. Q2: Abstain or ask clarifying question—never invent. Q3: Free generation with post-hoc footnotes unverified.

</details>


### Lecture 06 · Advanced and Enterprise RAG

Study graph, multi-hop, adaptive, and agentic retrieval together with tenancy, freshness, security, resilience, and cost.

#### One index, two audiences

A finance-only reviewer note explains an exception. A regular employee should see the approved policy, while a finance reviewer can see both sources.

Apply access controls before retrieval or context assembly and maintain separate verification cases per role. Review caches, indexing refresh, and logs because an answer can be correct while evidence crosses an audience boundary.

#### Read the mechanism

**Graph RAG**

Graph RAG combines knowledge graphs with retrieval so multi-hop relations traverse explicit edges. It helps when answers require chained entity relationships.

'Which vendor supplies part X used in product Y?' may need graph traversal, not one vector search.

**multi-hop retrieval**

Multi-hop retrieval gathers evidence across sequential lookups when no single passage contains the answer. Orchestration must avoid error propagation from early hops.

Finding budget owner requires hop one: project ID → department; hop two: department → approver.

**adaptive RAG**

Adaptive RAG chooses retrieval depth, query rewrite, or no retrieval based on question type and confidence. It saves cost on simple queries while going deep on hard ones.

Greetings skip retrieval; compliance questions trigger hybrid search plus rerank.

**authorization**

Authorization ensures retrieved and acted-upon data respects user permissions—not just authentication. RAG without authZ leaks restricted documents into answers.

An employee should not retrieve executive compensation docs via semantic search without role checks.

**freshness**

Freshness policies define acceptable document age, re-ingest cadence, and TTL for cached answers. Regulated domains often require sub-daily updates for policy corpora.

Benefits enrollment answers must exclude documents marked superseded after open enrollment ends.

#### Worked example

**Situation:** Enterprise RAG must support multi-hop questions, tenant isolation, and adaptive retrieval while staying within cost caps.

**Baseline:** Single-pass retrieve-then-generate for every query—expensive on simple lookups.

**Application:** Threat-model tenancy and injection, implement adaptive router (simple vs multi-hop), graph links for related policies, freshness timestamps, complete architecture studio exercise.

**Test cases:** (1) Normal: single-hop FAQ. (2) Boundary: multi-hop across leave and payroll policies. (3) Adversarial: cross-tenant ID in retrieved metadata.

**Measurement:** Cost per query type, tenant leak tests (zero tolerance), end-to-end accuracy on multi-hop slice.

**Design question:** What failure cannot be fixed by adding another retrieval hop?

**Predict before running.** Advanced orchestration cannot compensate for weak data, retrieval, authorization, or evaluation. Identify the input you would hold fixed and the observation that would change your decision.


In [ ]:
"""Lab 6.6: Advanced and Enterprise RAG"""

CHAPTER = "6.6"
print("chapter hook:", CHAPTER)
routes = {"simple": {"hops": 1, "cost": 1}, "multi": {"hops": 3, "cost": 4}}
def adaptive(query):
    return "multi" if "and also" in query or "depending on" in query else "simple"
queries = ["PTO cap?", "PTO cap and carryover depending on tenure"]
for q in queries:
    r = adaptive(q)
    print({"query": q, "route": r, "cost_units": routes[r]["cost"]})
print("---")
print("change one input above, predict output, re-run")


#### Interpret and apply

Complete the enterprise RAG architecture studio and threat model.

Change one input, record the prediction before running again, and compare with the unchanged output. If a result disappoints, record which intermediate stage first diverged.

1. Why cannot orchestration compensate for weak authorization or data?
2. When is adaptive retrieval worth the routing complexity?
3. What baseline always runs Graph RAG?

<details><summary>Check your reasoning</summary>

Q1: Bad ACLs, missing docs, or broken ingestion need foundational fixes. Q2: When simple queries dominate and multi-hop slice is small but costly. Q3: Maximum pipeline for every query regardless of complexity.

</details>


**Section checkpoint.** Define pass cases for authorized evidence, no relevant evidence, and a citation to the wrong passage.


## Course workshop

Build rankings with one shared candidate. Test how agreement affects rank before evaluating citation faithfulness separately.

Run the reference implementation, then run the checks below it.


In [ ]:
"""Fuse lexical and vector-like rankings with reciprocal rank fusion."""
LEXICAL = ["doc-expense", "doc-leave", "doc-security"]
VECTOR = ["doc-leave", "doc-expense", "doc-onboarding"]


def reciprocal_rank_fusion(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, document in enumerate(ranking, start=1):
            scores[document] = scores.get(document, 0.0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda item: item[1], reverse=True)


for document, score in reciprocal_rank_fusion([LEXICAL, VECTOR]):
    print(f"{score:.5f} {document}")


### Verify and extend

Run the assertions. Then change one assumption and add a check for the predicted behavior.


In [ ]:
ranked = reciprocal_rank_fusion([["shared", "lexical"], ["shared", "dense"]])
assert ranked[0][0] == "shared"
print('Workshop checks passed')


## Applied exercise for Lecture 05 — Evaluate grounded answers against an access boundary

Read the labeled cases before you run the cell. Predict which sources an employee should see. Then add a case for a user who has no authorized role. Check source IDs separately from answer text. The generator here returns a passage extract, so you can inspect the retrieval and citation behavior directly.


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Passage:
    source: str
    tags: set[str]
    text: str
    roles: set[str]


corpus = [
    Passage("travel", {"travel", "receipt"}, "Attach travel receipts to the expense form.", {"employee", "payroll"}),
    Passage("rates", {"payroll", "rate"}, "Use the current internal payroll rate table.", {"payroll"}),
]


def grounded_answer(query_terms, role):
    allowed = [passage for passage in corpus if role in passage.roles]
    ranked = sorted(allowed, key=lambda passage: len(query_terms & passage.tags), reverse=True)
    if not ranked or not query_terms & ranked[0].tags:
        return "No authorized matching evidence.", []
    evidence = ranked[0]
    return evidence.text, [evidence.source]


cases = [
    ({"travel", "receipt"}, "employee", ["travel"]),
    ({"payroll", "rate"}, "employee", []),
    ({"weather"}, "employee", []),
    ({"payroll", "rate"}, "payroll", ["rates"]),
]
for terms, role, expected_sources in cases:
    answer, sources = grounded_answer(terms, role)
    assert sources == expected_sources, (terms, role, sources)
    if not expected_sources:
        assert answer == "No authorized matching evidence."
    print(role, sorted(terms), answer, sources)


## Course project

Deliver an enterprise RAG system with authorization, hybrid retrieval, reranking, citations, and stage-specific evaluation.

Keep your baseline, results, and one failure case so someone else can review the decision you made.
